# 02 - Limpieza y transformacion para comparacion entre empresas

Este notebook prepara el dataset del foco 2 para dejarlo listo para ranking,
comparacion por composicion, side effects y especializacion por empresa.

In [1]:
from pathlib import Path
import sys

project_root = Path.cwd().resolve()
while not (project_root / "src").exists() and project_root != project_root.parent:
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd

from src.load_data import load_medicine_data
from src.enfoque_02_comparacion_empresas.cleaning import (
    COMPANY_COMPARISON_CLEAN_PATH,
    clean_company_comparison_data,
    extract_components,
    infer_therapeutic_areas,
    normalize_composition,
    normalize_manufacturer,
    split_side_effects,
)
from src.enfoque_02_comparacion_empresas.transform import (
    explode_side_effects,
    explode_therapeutic_areas,
)
from src.enfoque_02_comparacion_empresas.validation import full_quality_report

pd.set_option("display.max_colwidth", 120)

df_raw = load_medicine_data(download_if_missing=False)
print(f"Datos cargados: {df_raw.shape}")

Datos cargados: (11825, 9)


## 1. Decision tecnica: normalizar manufacturer y composition

La comparacion por empresa depende de dos llaves:

- `manufacturer_clean`: evita diferencias por espacios raros.
- `composition_key`: compara por ingredientes activos, sin dosis y sin depender del orden.

In [2]:
ejemplos_texto = df_raw.loc[[0, 1, 4], ["Manufacturer", "Composition"]].copy()
ejemplos_texto["manufacturer_clean"] = ejemplos_texto["Manufacturer"].map(normalize_manufacturer)
ejemplos_texto["composition_clean"] = ejemplos_texto["Composition"].map(normalize_composition)
ejemplos_texto["components_list"] = ejemplos_texto["Composition"].map(extract_components)
ejemplos_texto

,Manufacturer,Composition,manufacturer_clean,composition_clean,components_list
0,Roche Products India Pvt Ltd,Bevacizumab (400mg),Roche Products India Pvt Ltd,Bevacizumab (400mg),[Bevacizumab]
1,Glaxo SmithKline Pharmaceuticals Ltd,Amoxycillin (500mg) + Clavulanic Acid (125mg),Glaxo SmithKline Pharmaceuticals Ltd,Amoxycillin (500mg) + Clavulanic Acid (125mg),"[Amoxycillin, Clavulanic Acid]"
4,Cadila Pharmaceuticals Ltd,Ranitidine (150mg),Cadila Pharmaceuticals Ltd,Ranitidine (150mg),[Ranitidine]


## 2. Decision tecnica: `Uses` se transforma en areas terapeuticas

`Uses` no esta separado por delimitadores consistentes. En vez de forzar un
split fragil, se mapea a **areas terapeuticas amplias** a partir de keywords.
Esto es suficiente para estudiar especializacion por empresa.

In [3]:
ejemplos_uses = df_raw.loc[[0, 3, 6, 9], ["Medicine Name", "Uses"]].copy()
ejemplos_uses["therapeutic_areas"] = ejemplos_uses["Uses"].map(infer_therapeutic_areas)
ejemplos_uses

,Medicine Name,Uses,therapeutic_areas
0,Avastin 400mg Injection,Cancer of colon and rectum Non-small cell lung cancer Kidney cancer Brain tumor Ovarian cancer Cervical cancer,"[Oncology, Women's Health, Urology / Sexual Health]"
3,Ascoril LS Syrup,Treatment of Cough with mucus,[Respiratory / Allergy]
6,Avil 25 Tablet,Treatment of Allergic conditionsTreatment of Respiratory disease with excessive mucusTreatment of Skin conditions wi...,"[Pain / Inflammation, Respiratory / Allergy, Dermatology]"
9,Atarax 25mg Tablet,Treatment of AnxietyTreatment of Skin conditions with inflammation & itching,"[Pain / Inflammation, Dermatology]"


## 3. Decision tecnica: `Side_effects` se separa a lista

La columna raw concatena efectos usando mayusculas como frontera implicita.
La transformacion deja una lista para poder explotar luego empresa x efecto.

In [4]:
ejemplos_fx = df_raw.loc[[0, 1, 3], ["Medicine Name", "Side_effects"]].copy()
ejemplos_fx["side_effects_list"] = ejemplos_fx["Side_effects"].map(split_side_effects)
ejemplos_fx

,Medicine Name,Side_effects,side_effects_list
0,Avastin 400mg Injection,Rectal bleeding Taste change Headache Nosebleeds Back pain Dry skin High blood pressure Protein in urine Inflammatio...,"[Rectal bleeding, Taste change, Headache, Nosebleeds, Back pain, Dry skin, High blood pressure, Protein in urine, In..."
1,Augmentin 625 Duo Tablet,Vomiting Nausea Diarrhea Mucocutaneous candidiasis,"[Vomiting, Nausea, Diarrhea, Mucocutaneous candidiasis]"
3,Ascoril LS Syrup,Nausea Vomiting Diarrhea Upset stomach Stomach pain Allergic reaction Dizziness Headache Rash Hives Tremors Palpitat...,"[Nausea, Vomiting, Diarrhea, Upset stomach, Stomach pain, Allergic reaction, Dizziness, Headache, Rash, Hives, Tremo..."


## 4. Ejecutar pipeline completo

In [5]:
df_clean = clean_company_comparison_data(df_raw, save=True)
print(f"Shape raw   : {df_raw.shape}")
print(f"Shape limpio: {df_clean.shape}")
print(f"CSV guardado en: {COMPANY_COMPARISON_CLEAN_PATH}")

df_clean.head(3)

Shape raw   : (11825, 9)
Shape limpio: (11741, 19)
CSV guardado en: C:\Users\cesar\OneDrive\Desktop\DuocUC\SCY1101-Exp1-Drug-Data-Analysis\data\processed\manufacturer_comparison_clean.csv


,Medicine Name,Composition,Uses,Side_effects,Image URL,Manufacturer,Excellent Review %,Average Review %,Poor Review %,manufacturer_clean,composition_clean,components_list,composition_key,side_effects_list,therapeutic_areas,n_components,n_side_effects,review_sum,review_balance
0,Avastin 400mg Injection,Bevacizumab (400mg),Cancer of colon and rectum Non-small cell lung cancer Kidney cancer Brain tumor Ovarian cancer Cervical cancer,Rectal bleeding Taste change Headache Nosebleeds Back pain Dry skin High blood pressure Protein in urine Inflammatio...,"https://onemg.gumlet.io/l_watermark_346,w_480,h_480/a_ignore,w_480,h_480,c_fit,q_auto,f_auto/f5a26c491e4d48199ab116a...",Roche Products India Pvt Ltd,22,56,22,Roche Products India Pvt Ltd,Bevacizumab (400mg),[Bevacizumab],Bevacizumab,"[Rectal bleeding, Taste change, Headache, Nosebleeds, Back pain, Dry skin, High blood pressure, Protein in urine, In...","[Oncology, Women's Health, Urology / Sexual Health]",1,9,100,0
1,Augmentin 625 Duo Tablet,Amoxycillin (500mg) + Clavulanic Acid (125mg),Treatment of Bacterial infections,Vomiting Nausea Diarrhea Mucocutaneous candidiasis,"https://onemg.gumlet.io/l_watermark_346,w_480,h_480/a_ignore,w_480,h_480,c_fit,q_auto,f_auto/wy2y9bdipmh6rgkrj0zm.jpg",Glaxo SmithKline Pharmaceuticals Ltd,47,35,18,Glaxo SmithKline Pharmaceuticals Ltd,Amoxycillin (500mg) + Clavulanic Acid (125mg),"[Amoxycillin, Clavulanic Acid]",Amoxycillin + Clavulanic Acid,"[Vomiting, Nausea, Diarrhea, Mucocutaneous candidiasis]",[Infections],2,4,100,29
2,Azithral 500 Tablet,Azithromycin (500mg),Treatment of Bacterial infections,Nausea Abdominal pain Diarrhea,"https://onemg.gumlet.io/l_watermark_346,w_480,h_480/a_ignore,w_480,h_480,c_fit,q_auto,f_auto/cropped/kqkouvaqejbyk47...",Alembic Pharmaceuticals Ltd,39,40,21,Alembic Pharmaceuticals Ltd,Azithromycin (500mg),[Azithromycin],Azithromycin,"[Nausea, Abdominal pain, Diarrhea]",[Infections],1,3,100,18


## 5. Validacion post-limpieza

In [6]:
report = full_quality_report(df_raw)
print(report)

print("\nResumen de columnas derivadas:")
print(df_clean[["n_components", "n_side_effects", "review_balance"]].describe().round(2))

{'shape': (11825, 9), 'duplicates': 84, 'manufacturer_blank': 0, 'composition_blank': 0, 'review_inconsistencies': 0, 'manufacturer_unique': 759, 'null_summary': {'Medicine Name': 0, 'Composition': 0, 'Uses': 0, 'Side_effects': 0, 'Manufacturer': 0, 'Excellent Review %': 0, 'Average Review %': 0, 'Poor Review %': 0}}

Resumen de columnas derivadas:
       n_components  n_side_effects  review_balance
count      11741.00        11741.00        11741.00
mean           1.53            6.92           12.79
std            0.77            4.31           45.64
min            1.00            1.00         -100.00
25%            1.00            4.00          -12.00
50%            1.00            6.00           13.00
75%            2.00            9.00           42.00
max            9.00           36.00          100.00


In [7]:
columnas_clave = [
    "Medicine Name",
    "manufacturer_clean",
    "composition_clean",
    "composition_key",
    "components_list",
    "therapeutic_areas",
    "side_effects_list",
    "n_components",
    "n_side_effects",
    "review_balance",
]

df_clean[columnas_clave].head(5)

,Medicine Name,manufacturer_clean,composition_clean,composition_key,components_list,therapeutic_areas,side_effects_list,n_components,n_side_effects,review_balance
0,Avastin 400mg Injection,Roche Products India Pvt Ltd,Bevacizumab (400mg),Bevacizumab,[Bevacizumab],"[Oncology, Women's Health, Urology / Sexual Health]","[Rectal bleeding, Taste change, Headache, Nosebleeds, Back pain, Dry skin, High blood pressure, Protein in urine, In...",1,9,0
1,Augmentin 625 Duo Tablet,Glaxo SmithKline Pharmaceuticals Ltd,Amoxycillin (500mg) + Clavulanic Acid (125mg),Amoxycillin + Clavulanic Acid,"[Amoxycillin, Clavulanic Acid]",[Infections],"[Vomiting, Nausea, Diarrhea, Mucocutaneous candidiasis]",2,4,29
2,Azithral 500 Tablet,Alembic Pharmaceuticals Ltd,Azithromycin (500mg),Azithromycin,[Azithromycin],[Infections],"[Nausea, Abdominal pain, Diarrhea]",1,3,18
3,Ascoril LS Syrup,Glenmark Pharmaceuticals Ltd,Ambroxol (30mg/5ml) + Levosalbutamol (1mg/5ml) + Guaifenesin (50mg/5ml),Ambroxol + Guaifenesin + Levosalbutamol,"[Ambroxol, Levosalbutamol, Guaifenesin]",[Respiratory / Allergy],"[Nausea, Vomiting, Diarrhea, Upset stomach, Stomach pain, Allergic reaction, Dizziness, Headache, Rash, Hives, Tremo...",3,14,-11
4,Aciloc 150 Tablet,Cadila Pharmaceuticals Ltd,Ranitidine (150mg),Ranitidine,[Ranitidine],[Gastrointestinal],"[Headache, Diarrhea, Gastrointestinal disturbance]",1,3,5


## 6. Transformaciones listas para el analisis

A partir del dataset limpio se construyen dos vistas utiles:

- `medicine x side_effect`
- `medicine x therapeutic_area`

In [ ]:
df_side = explode_side_effects(df_clean)
df_area = explode_therapeutic_areas(df_clean)

print(f"Vista medicine x side_effect      : {df_side.shape}")
print(f"Vista medicine x therapeutic_area : {df_area.shape}")

In [ ]:
df_side[["Medicine Name", "manufacturer_clean", "side_effect"]].head(8)

In [ ]:
df_area[["Medicine Name", "manufacturer_clean", "therapeutic_area"]].head(8)

## 7. Resultado del notebook 02

El output principal queda en:

- `data/processed/manufacturer_comparison_clean.csv`

Ese archivo ya contiene todo lo necesario para el notebook 03: volumen por
empresa, composicion comparable, efectos secundarios, areas terapeuticas y
metricas sinteticas de review.